# E-commerce Customer Retention Analysis

## Part 3 — Natural Language Processing and Sentiment Analysis

### Objective

Analyze customer reviews using Natural Language Processing (NLP) techniques to identify sentiment patterns associated with customer satisfaction and customer recurrence. The objective is to extract valuable information from textual data and complement the predictive models developed in the previous stages of the project.

### Dataset

Customer reviews from the Olist Brazilian E-commerce Dataset.

### Technologies

- Python
- Pandas
- NumPy
- NLTK
- Gensim
- Word2Vec
- TensorFlow / Keras
- Matplotlib

# Database Integration

The first section of this notebook reproduces the **data integration process** presented in the previous notebook, where the original Olist datasets are merged into a single analytical database.

This integrated dataset is then used to prepare the final modeling dataset and to implement the **Deep Learning** model presented in the following sections.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import sys
!{sys.executable} -m pip install -U pandas-profiling[notebook]

In [ ]:
# Install ydata-profiling if necessary

try:
    import ydata_profiling as pp
    from ydata_profiling import ProfileReport
except ModuleNotFoundError:
    !pip install -q ydata-profiling
    import ydata_profiling as pp
    from ydata_profiling import ProfileReport

In [ ]:
from datetime import datetime
from datetime import timedelta

In [ ]:
# Install required packages (Google Colab / Jupyter)

import sys
import subprocess

packages = [
    "ydata-profiling",
    "gensim",
    "torch",
    "tqdm"
]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

In [ ]:
import functools
import gzip
import pandas as pd
import tempfile
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

from gensim import corpora
from gensim.parsing import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, accuracy_score
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm, trange

In [ ]:
# Data source: Olist Brazilian E-commerce Dataset
# CSV files are loaded directly from this GitHub repository.

import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/Jorgelina-Etchevest/ecommerce-customer-retention-analysis/main"

df1 = pd.read_csv(f"{BASE_URL}/olist_customers_dataset.csv")
df2 = pd.read_csv(f"{BASE_URL}/olist_order_items_dataset.csv")
df3 = pd.read_csv(f"{BASE_URL}/olist_order_payments_dataset.csv")
df4 = pd.read_csv(f"{BASE_URL}/olist_order_reviews_dataset.csv")
df5 = pd.read_csv(f"{BASE_URL}/olist_orders_dataset.csv")
df6 = pd.read_csv(f"{BASE_URL}/olist_products_dataset.csv")
df7 = pd.read_csv(f"{BASE_URL}/olist_sellers_dataset.csv")
df8 = pd.read_csv(f"{BASE_URL}/product_category_name_translation.csv")

In [ ]:
df_merged = pd.merge(df2, df3, how="outer", on="order_id")
df_merged = pd.merge(df_merged, df4, how="outer", on="order_id")
df_merged = pd.merge(df_merged, df5, how="outer", on="order_id")
df_merged = pd.merge(df_merged, df1, how="outer", on="customer_id")
df_merged = pd.merge(df_merged, df6, how="outer", on="product_id")
df_merged = pd.merge(df_merged, df7, how="outer", on="seller_id")
df_merged = pd.merge(df_merged, df8, how="outer", on="product_category_name")

df_merged.sample(6)

In [ ]:
df_merged.shape

In [ ]:
df_merged.columns

In [ ]:
df_merged.info()

# Data Cleaning and Exploratory Analysis

The objective of this analysis is to examine the **customer reviews** left on the Olist platform. The textual analysis is performed using the **original reviews in Portuguese**, while only the most relevant results are translated into English.

As a preprocessing step, duplicate **`order_id`** entries for the same customer were removed. These duplicate records correspond to multiple items purchased within the **same order**, rather than multiple purchases. Therefore, retaining only one record per customer-order combination ensures that each purchase is counted only once. Missing values were also removed where appropriate.

Since the primary objective is to analyze **customer return behavior**, we first examine the distribution of returning customers. Among the **9,333 customers** who left a review, only **1,484** made a subsequent purchase on the Olist platform, highlighting a substantial class imbalance in the dataset.


In [ ]:
frec = df_merged[['customer_unique_id','order_id']].drop_duplicates()

In [ ]:
df_merged = df_merged.dropna()

In [ ]:
clientes_con_review = df_merged[df_merged['review_comment_message'].notnull()]

In [ ]:
total_clientes_review = clientes_con_review['customer_unique_id'].nunique()

print(f"El número total de clientes que hicieron una reseña es: {total_clientes_review}")

In [ ]:
frec = clientes_con_review.groupby('customer_unique_id')['order_id'].count()
frec = frec.reset_index()
frec.columns = ['customer_unique_id', 'frecuencia']
frec.info()

In [ ]:
frec_2 = frec[frec['frecuencia'] >= 2]
print("La cantidad de clientes que volvieron a comprar es =",frec_2.shape[0])

In [ ]:
#Línea de código para pandas no trunque el texto de la review
pd.set_option('display.max_colwidth', None)

In [ ]:
print(clientes_con_review[['customer_unique_id', 'review_comment_message']].head(5))

# Dataset Preparation

The dataset was prepared by retaining only the variables required for the sentiment analysis: **customer identifier**, **review text**, and **review score**.

In the Olist platform, customer satisfaction is measured using a **five-point rating scale**, where review scores range from **1 to 5**.

A new target variable, **`sentiment`**, was created based on the review score:
- **Negative:** review scores from **1 to 3**
- **Positive:** review scores from **4 to 5**

Among all customers who submitted a review, **68%** were classified as **Positive**, while the remaining **32%** were classified as **Negative**. This distribution indicates a moderate class imbalance that should be considered during model training and evaluation.


In [ ]:
df = clientes_con_review[['customer_id', 'review_comment_message', 'review_score']]

In [ ]:
df['sent'] = np.where(df['review_score'] <= 3, 'negativo', 'positivo')
print(df[['review_score', 'sent']].head(10))

In [ ]:
df_dataset = pd.DataFrame.from_dict(df[:]).rename(columns={'review_comment_message':'reseña','sent':'sentimiento'})

In [ ]:
df_dataset.head(15)

In [ ]:
df_sentimiento = df_dataset[['reseña', 'sentimiento']]

In [ ]:
df_sentimiento.head()

In [ ]:
df_sentimiento['sentimiento'].value_counts()

In [ ]:
proporciones = df_sentimiento['sentimiento'].value_counts(normalize=True)

# Mostrar las proporciones
print(proporciones)

# Visualizar las proporciones
proporciones.plot(kind='bar')
plt.title('REVIEW: Proporciones de Sentimientos')
plt.xlabel('Sentimiento')
plt.ylabel('Proporción')
plt.show()

# Deep Learning Models

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## Preprocessing

Before training a neural network, the input features must be transformed into a numerical representation. In this project, the only input feature is the **customer review text**.

As part of the preprocessing stage, the text data undergoes several standard normalization steps using the **preprocessing module**. These transformations prepare the textual data for the neural network by converting the reviews into a structured numerical format.

Finally, each unique word is **tokenized** and mapped to an **integer index** in a vocabulary, allowing the text to be processed by the Deep Learning model.

In [ ]:
class RawDataProcessor:
    def __init__(self,
                 dataset,
                 ignore_header=True,
                 filters=None,
                 vocab_size=50000):
        if filters:
            self.filters = filters
        else:
            self.filters = [
                lambda s: s.lower(),
                preprocessing.strip_tags,
                preprocessing.strip_punctuation,
                preprocessing.strip_multiple_whitespaces,
                preprocessing.strip_numeric,
                preprocessing.remove_stopwords,
                preprocessing.strip_short,
            ]

        # Crea un diccionario basado en todas las reseñas (con el preprocesamiento correspondiente)
        self.dictionary = corpora.Dictionary(
            dataset["reseña"].map(self._preprocess_string).tolist()
        )
        # Filtrar el diccionario con palabras extremas
        self.dictionary.filter_extremes(no_below=2, no_above=1, keep_n=vocab_size)

        # Indices continuos después de haber eliminado algunas palabras
        self.dictionary.compactify()

        # Agregar un par de tokens especiales
        self.dictionary.patch_with_special_tokens({
            "[PAD]": 0,
            "[UNK]": 1
        })
        self.idx_to_target = sorted(dataset["sentimiento"].unique())
        self.target_to_idx = {t: i for i, t in enumerate(self.idx_to_target)}


    def _preprocess_string(self, string):
        return preprocessing.preprocess_string(string, filters=self.filters)

    def _sentence_to_indices(self, sentence):
        return self.dictionary.doc2idx(sentence, unknown_word_index=1)

    def encode_data(self, data):
        return self._sentence_to_indices(self._preprocess_string(data))

    def encode_target(self, target):
        return self.target_to_idx[target]

    def __call__(self, item):
        if isinstance(item["review_comment_message"], str):
            data = self.encode_data(item["review_comment_message"])
        else:
            data = [self.encode_data(d) for d in item["review_comment_message"]]

        if isinstance(item["sent"], str):
            target = self.encode_target(item["sent"])
        else:
            target = [self.encode_target(t) for t in item["sent"]]

        return {
            "review_comment_message": data,
            "sent": target
        }

In [ ]:
processor = RawDataProcessor(df_sentimiento)

dataset_transform = df_sentimiento.apply(lambda row: processor({'review_comment_message': row['reseña'], 'sent': row['sentimiento']}), axis=1)

for idx, sample in enumerate(dataset_transform):
    print("review_comment_message")
    print(sample["review_comment_message"])
    print("sent:", sample["sent"])
    print("=" * 50)

    if idx == 2:
        break

After preprocessing, the dataset was divided into **training** and **test** sets. **Twenty percent** of the observations were reserved for the **test set**, while the remaining **80%** were used to train the Deep Learning model.

In [ ]:
train_indices, test_indices = train_test_split(dataset_transform.index, test_size=0.2, random_state=123)

train_dataset_transform = dataset_transform.loc[train_indices].reset_index(drop=True)
test_dataset_transform = dataset_transform.loc[test_indices].reset_index(drop=True)

In [ ]:
# Creamos los datasets sin preprocesar para visualizar
test_dataset_notransform = df_sentimiento.loc[test_indices].reset_index(drop=True)
train_dataset_notransform = df_sentimiento.loc[train_indices].reset_index(drop=True)

print(f"Datasets loaded with {len(train_dataset_transform)} training elements and {len(test_dataset_transform)} test elements.")
print()
print(f"Sample train element:\n{train_dataset_notransform.iloc[1]}")
print()
print(f"Sample train element with preprocess:\n{train_dataset_transform.iloc[1]}")

Since the input data consists of **sequences of words** represented by their corresponding **vocabulary indices**, the sequences may have different lengths.

When creating mini-batches, **PyTorch's `DataLoader`** requires all sequences within a batch to have the **same length** so they can be efficiently combined into a fixed-size tensor. Therefore, the sequences must be appropriately **padded or truncated** before being passed to the model.

In [ ]:
class PadSequences:
    def __init__(self, pad_value=0, max_length=None, min_length=1):
        assert max_length is None or min_length <= max_length
        self.pad_value = pad_value
        self.max_length = max_length
        self.min_length = min_length

    def __call__(self, items):
        data, target = list(zip(*[(item["review_comment_message"], item["sent"]) for item in items]))
        seq_lengths = [len(d) for d in data]

        if self.max_length:
            max_length = self.max_length
            seq_lengths = [min(self.max_length, l) for l in seq_lengths]
        else:
            max_length = max(self.min_length, max(seq_lengths))

        data = [d[:l] + [self.pad_value] * (max_length - l)
                for d, l in zip(data, seq_lengths)]

        return {
            "review_comment_data": torch.LongTensor(data),
            "sent": torch.FloatTensor(target)
        }

In [ ]:
pad_sequences = PadSequences()
train_loader = DataLoader(train_dataset_transform, batch_size=128, shuffle=True,
                          collate_fn=pad_sequences, drop_last=False)
test_loader = DataLoader(test_dataset_transform, batch_size=128, shuffle=False,
                         collate_fn=pad_sequences, drop_last=False)

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

## The Model

For the classification task, we implement a **Multilayer Perceptron (MLP)** with **two hidden layers**.

The code block below defines a neural network model named **`IMDBReviewsClassifier`**, designed for **binary text classification** tasks, such as determining whether a customer review expresses a **positive** or **negative** sentiment.

In [ ]:
class IMDBReviewsClassifier(nn.Module):
    def __init__(self,
                 pretrained_embeddings_path,
                 dictionary,
                 vector_size,
                 freeze_embedings):
        super().__init__()
        embeddings_matrix = torch.randn(len(dictionary), vector_size)
        embeddings_matrix[0] = torch.zeros(vector_size)
        with open("glove.6B.50d.txt", "r", encoding="utf-8") as fh:
            for line in fh:
                word, vector = line.strip().split(None, 1)
                if word in dictionary.token2id:
                    embeddings_matrix[dictionary.token2id[word]] = \
                        torch.FloatTensor([float(n) for n in vector.split()])
        self.embeddings = nn.Embedding.from_pretrained(embeddings_matrix,
                                                       freeze=freeze_embedings,
                                                       padding_idx=0)
        self.hidden1 = nn.Linear(vector_size, 128)
        self.hidden2 = nn.Linear(128, 128)
        self.output = nn.Linear(128, 1)
        self.vector_size = vector_size

    def forward(self, x):
        x = self.embeddings(x)
        x = torch.mean(x, dim=1)
        x = F.relu(self.hidden1(x))
        x = F.relu(self.hidden2(x))
        x = torch.sigmoid(self.output(x))
        return x

## Neural Network Classifier

In [ ]:
model = IMDBReviewsClassifier("glove.6B.50d.txt", processor.dictionary, 50, True)
loss = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
for epoch in trange(3):
    model.train()
    running_loss = []
    for idx, batch in enumerate(tqdm(train_loader)):
        optimizer.zero_grad()
        output = model(batch["review_comment_data"])
        loss_value = loss(output, batch["sent"].view(-1, 1))
        loss_value.backward()
        optimizer.step()
        running_loss.append(loss_value.item())
    print(sum(running_loss) / len(running_loss), "epoch", epoch+1)

    model.eval()
    running_loss = []
    targets = []
    predictions = []
    for batch in tqdm(test_loader):
        output = model(batch["review_comment_data"])
        running_loss.append(
            loss(output, batch["sent"].view(-1, 1)).item()
        )
        targets.extend(batch["sent"].numpy())
        predictions.extend(output.squeeze().detach().numpy())
    print(sum(running_loss) )

In [ ]:
th = .5
dataset_for_visual = pd.DataFrame.from_dict(test_dataset_notransform[:]).rename(columns={'review_comment_message':'reseña','sent':'sentimiento'})
dataset_for_visual['sent'] = targets
dataset_for_visual['prediction'] = predictions
dataset_for_visual['pred_sentiment'] = 0.
dataset_for_visual.loc[dataset_for_visual.prediction >= th ,'pred_sentiment'] = 1.
dataset_for_visual.head()

## Evaluation Metrics

Model performance was evaluated by comparing the **predicted sentiment** with the **true customer sentiment**. The classifier achieved an **accuracy of 0.70**.

- **Class 0:** Negative Reviews
- **Class 1:** Positive Reviews

Given the business objective, every review predicted as **Negative** could trigger a customer service or retention action, such as a promotional offer, discount or follow-up communication.

Therefore, the priority is to **maximize Precision for the Negative class**, thereby **minimizing false positives**. In this context, a false positive occurs when a truly **Positive** review is incorrectly classified as **Negative**.

These errors would lead the company to allocate discounts or retention resources to customers who were actually satisfied, increasing costs without generating additional value.

In [ ]:
accuracy = accuracy_score(dataset_for_visual.sent, dataset_for_visual.pred_sentiment)
print(accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
print("\nMatriz de confusión para el Modelo: ")
print(confusion_matrix(dataset_for_visual.sent, dataset_for_visual.pred_sentiment))

In [ ]:
ConfusionMatrixDisplay.from_predictions(dataset_for_visual.sent, dataset_for_visual.pred_sentiment,normalize='true')

In [ ]:
print(classification_report(dataset_for_visual.sent,dataset_for_visual.pred_sentiment, target_names=["NEGATIVA","POSITIVA"]))

## Analysis of Misclassified Reviews

### False Negatives

This section analyzes **false negatives**, i.e., reviews that were **classified as Negative by the model but were actually Positive** according to the customer's rating.

A closer inspection of these reviews reveals that many customers mention aspects they **did not like** or features that **did not meet their expectations**, while still expressing an overall positive opinion of the product or service. Consequently, they assigned a satisfaction score of **4 or 5**, leading to a **Positive** label in the dataset.

These examples illustrate the challenges of **sentiment analysis**, where reviews often contain **mixed or nuanced sentiments**. Although negative expressions are present, the overall sentiment remains positive, making these cases particularly difficult for the classifier.

**Examples:**

> *"The folder is great, and I really liked it. The inner pocket is a bit wide and doesn't have an elastic band, so the items stored there tend to fall out during transport. Other than that, it's perfect."*

> *"I can't give a full opinion yet because I only started using it this week. I was disappointed because the bottles arrived crushed... Thank you."*

> *"When purchasing the electric grill pan, the product photos made us believe that its depth was the same as the outer height of the pan. In reality, it is much shallower than expected, which was disappointing."*

In [ ]:
false_negatives = dataset_for_visual[(dataset_for_visual.pred_sentiment == 0) & (dataset_for_visual.sent == 1)]
print(false_negatives)

In [ ]:
dataset_for_visual.iloc[259].reseña

In [ ]:
dataset_for_visual.iloc[1758].reseña

In [ ]:
dataset_for_visual.iloc[2091].reseña

# Conclusions

Natural Language Processing complemented the structured Machine Learning models by incorporating information contained in customer reviews.

The neural network successfully identified sentiment patterns associated with customer satisfaction, achieving an overall accuracy of approximately **70%**.

The analysis of misclassified reviews revealed that many prediction errors occurred in reviews containing both positive and negative statements. Although customers mentioned specific complaints, they still assigned high satisfaction scores, highlighting the complexity of human language and sentiment interpretation.

This analysis demonstrates how textual data can provide valuable business insights that complement traditional transactional data, supporting more informed customer retention strategies.